# 🏛️ Apache Iceberg & Trino Modern Lakehouse Deep-Dive
## ACID Snapshots, Partition Pruning, Time-Travel Queries & In-Place Schema Evolution

This notebook demonstrates the core storage and query mechanics of the modern Lakehouse:
1. **Catalog & Namespace Management** via PyIceberg REST Catalog.
2. **Explicit Table Partitioning** (`day(tpep_pickup_datetime)`) enabling Trino query partition pruning.
3. **Columnar Ingestion** with DuckDB and PyArrow into immutable Parquet files.
4. **Metadata Anatomy Inspection** via Trino internal system tables (`$snapshots`, `$manifests`, `$partitions`, `$files`).
5. **Distributed ANSI SQL Querying & Partition Pruning** in Trino.
6. **Reproducible Time-Travel Querying** using `FOR VERSION AS OF` and `FOR TIMESTAMP AS OF`.
7. **Non-Destructive In-Place Schema Evolution** (adding and renaming columns with zero data rewrites).
8. **Lakehouse Table Maintenance** (snapshot expiration and metadata hygiene).

In [ ]:
import os
import time
from datetime import datetime

CATALOG_NAME = os.getenv("CATALOG_NAME", "lakehouse-poc")
LAKEHOUSE_DATA_PATH = os.getenv("LAKEHOUSE_DATA_PATH", "/lakehouse-poc")
print(f"Catalog: {CATALOG_NAME} | Data path: {LAKEHOUSE_DATA_PATH}")

### 1. Catalog & Namespace Initialization
Idempotently clean and provision the `jupyter` namespace inside the Iceberg REST Catalog.

In [ ]:
from pyiceberg.catalog import load_catalog
from pyiceberg.table import Table
from pyiceberg.schema import Schema
from pyiceberg.types import (
    DoubleType, StringType, TimestampType, NestedField, 
    IntegerType, BooleanType, LongType
)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform, MonthTransform, IdentityTransform
from pyiceberg.table.sorting import SortOrder, SortField

catalog = load_catalog(name=CATALOG_NAME)
NAMESPACE = "jupyter"
TABLE_NAME = "nyc_taxi_yellow_tripdata"
FULL_TABLE_NAME = f"{NAMESPACE}.{TABLE_NAME}"

def create_namespace(ns):
    current_namespaces = { x[0] for x in catalog.list_namespaces() }
    if ns in current_namespaces:
        print(f"Namespace '{ns}' exists. Clearing existing tables...")
        for table in catalog.list_tables(namespace=ns):
            catalog.drop_table(table)
        catalog.drop_namespace(namespace=ns)
    catalog.create_namespace(
        namespace=ns,
        properties=dict(customer_name=ns, owner="lakehouse-admin@lakehouse.poc")
    )
    print(f"Catalog database created for namespace <{ns}>")

create_namespace(NAMESPACE)

### 2. Define Schema & Explicit Table Partitioning Specification
We define explicit table partitioning using Iceberg's `DayTransform` on `tpep_pickup_datetime`.
When Trino executes analytical queries filtering on pickup dates, it inspects manifest lower/upper bounds
and prunes non-matching partitions **without reading underlying Parquet data files**.

In [ ]:
columns = [
    ("VendorID", IntegerType()),
    ("tpep_pickup_datetime", TimestampType()),
    ("tpep_dropoff_datetime", TimestampType()),
    ("passenger_count", LongType()),
    ("trip_distance", DoubleType()),
    ("RatecodeID", LongType()),
    ("store_and_fwd_flag", StringType()),
    ("PULocationID", IntegerType()),
    ("DOLocationID", IntegerType()),
    ("payment_type", LongType()),
    ("fare_amount", DoubleType()),
    ("extra", DoubleType()),
    ("mta_tax", DoubleType()),
    ("tip_amount", DoubleType()),
    ("tolls_amount", DoubleType()),
    ("improvement_surcharge", DoubleType()),
    ("total_amount", DoubleType()),
    ("congestion_surcharge", DoubleType()),
    ("Airport_fee", DoubleType()),
]

iceberg_fields = []
for index, (col_name, field_type) in enumerate(columns):
    iceberg_fields.append(
        NestedField(field_id=index + 1, name=col_name, field_type=field_type, required=False)
    )
iceberg_schema = Schema(*iceberg_fields)

# Explicit table partition specification: partition by day of tpep_pickup_datetime
# source_id=2 corresponds to tpep_pickup_datetime
partition_spec = PartitionSpec(
    PartitionField(field_id=1000, source_id=2, transform=DayTransform(), name="pickup_day"),
)

# Create table in Iceberg REST Catalog with explicit partition specification
table = catalog.create_table(
    identifier=FULL_TABLE_NAME,
    location=f"s3a://warehouse/{NAMESPACE}/{TABLE_NAME}",
    schema=iceberg_schema,
    partition_spec=partition_spec,
)

print(f"Iceberg partitioned table created: {table.identifier}")
print(f"Partition specification: {table.spec()}")

### 3. Verify Table Creation via PyIceberg Catalog

In [ ]:
ns_names = {x[0] for x in catalog.list_namespaces()}
assert NAMESPACE in ns_names
table_names = {x[1] for x in catalog.list_tables(NAMESPACE)}
assert TABLE_NAME in table_names
print("Namespace and table confirmed in catalog.")

### 4. Extract Dataset using DuckDB into PyArrow Table

In [ ]:
import duckdb

data_file = f"{LAKEHOUSE_DATA_PATH}/datasets/yellow_tripdata_2024-01.parquet"
duckdb_data = duckdb.read_parquet(data_file)
arrow_table = duckdb_data.to_arrow_table()

print(f"Extracted {arrow_table.num_rows:,} rows across {arrow_table.num_columns} columns into Arrow memory.")

### 5. Append Dataset to Iceberg Table (Snapshot 1 Commit)
This atomic append writes Parquet data files to MinIO object storage, generates an Avro manifest file,
creates an Avro manifest list, and commits **Snapshot 1** to the catalog database.

In [ ]:
# Commit Snapshot 1
start_time = time.time()
table.append(arrow_table)
commit_duration = round(time.time() - start_time, 2)

current_snap = table.current_snapshot()
snapshot_1_id = current_snap.snapshot_id
snapshot_1_timestamp = current_snap.timestamp_ms
snapshot_1_dt = datetime.utcfromtimestamp(snapshot_1_timestamp / 1000).isoformat()

print(f"Snapshot 1 committed in {commit_duration}s:")
print(f"  Snapshot ID : {snapshot_1_id}")
print(f"  Timestamp   : {snapshot_1_timestamp} ({snapshot_1_dt} UTC)")
print(f"  Summary     : {current_snap.summary}")

### 6. Inspect Apache Iceberg Metadata Anatomy via Trino Internal Tables
Apache Iceberg provides internal system tables that expose metadata directly via ANSI SQL:
- `<table>$snapshots`: History of commits, snapshot IDs, parents, and manifest lists.
- `<table>$manifests`: Individual manifest files tracking data files and partition bounds.
- `<table>$partitions`: Partition summary records and file counts per partition.
- `<table>$files`: Data files on storage, format, record counts, and byte sizes.

In [ ]:
from trino.dbapi import connect
import pandas as pd

conn = connect(
    host="trino",
    port=8080,
    user="lakehouse-analyst",
    catalog=CATALOG_NAME,
    schema=NAMESPACE,
)
cur = conn.cursor()

# Query snapshots metadata table
cur.execute(f'SELECT snapshot_id, parent_id, committed_at, operation, manifest_list FROM "{NAMESPACE}"."{TABLE_NAME}$snapshots"')
snapshots_df = pd.DataFrame(cur.fetchall(), columns=["snapshot_id", "parent_id", "committed_at", "operation", "manifest_list"])
print("=== Iceberg Snapshots Metadata ===")
print(snapshots_df)

# Query manifests metadata table
cur.execute(f'SELECT path, length, partition_spec_id, added_data_files_count FROM "{NAMESPACE}"."{TABLE_NAME}$manifests"')
manifests_df = pd.DataFrame(cur.fetchall(), columns=["path", "length", "spec_id", "added_files"])
print("\n=== Iceberg Manifests Metadata ===")
print(manifests_df.head())

### 7. Trino Distributed ANSI SQL & Partition Pruning Verification
When executing a date-bounded query on `tpep_pickup_datetime`, Trino prunes partitions
using the `pickup_day` partition specification without scanning unrelated Parquet files.

In [ ]:
# Verify total row count
cur.execute(f'SELECT COUNT(*) FROM "{NAMESPACE}"."{TABLE_NAME}"')
row_count = cur.fetchone()[0]
print(f"Total row count in Trino: {row_count:,} rows")
assert row_count == 2964624

# Targeted date-range query leveraging partition pruning
query = f'''
SELECT 
    DATE(tpep_pickup_datetime) AS pickup_date,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM "{NAMESPACE}"."{TABLE_NAME}"
WHERE tpep_pickup_datetime >= parse_datetime('2024-01-01 00:00:00', 'YYYY-MM-dd HH:mm:ss')
  AND tpep_pickup_datetime < parse_datetime('2024-01-03 00:00:00', 'YYYY-MM-dd HH:mm:ss')
GROUP BY DATE(tpep_pickup_datetime)
ORDER BY pickup_date
'''
cur.execute(query)
date_agg_df = pd.DataFrame(cur.fetchall(), columns=["pickup_date", "trip_count", "avg_fare", "avg_distance"])
print("\n=== Partition-Pruned Aggregation Result ===")
print(date_agg_df)

### 8. Explain Plan: Demonstrating Trino Manifest-Level Partition Pruning
Using `EXPLAIN (TYPE DISTRIBUTED)`, we inspect Trino's query execution plan to confirm
that partition predicates are pushed down directly to the Iceberg metadata reader.

In [ ]:
explain_query = f'''
EXPLAIN (TYPE DISTRIBUTED)
SELECT COUNT(*) FROM "{NAMESPACE}"."{TABLE_NAME}"
WHERE tpep_pickup_datetime >= parse_datetime('2024-01-01 00:00:00', 'YYYY-MM-dd HH:mm:ss')
  AND tpep_pickup_datetime < parse_datetime('2024-01-02 00:00:00', 'YYYY-MM-dd HH:mm:ss')
'''
cur.execute(explain_query)
explain_lines = [r[0] for r in cur.fetchall()]
print("=== Trino Distributed Plan (Partition Pruning) ===")
print("\n".join(explain_lines[:15]))

### 9. Time-Travel Demonstration: Create Snapshot 2
To demonstrate reproducible time-travel, we append a second batch of records (a slice of 1,000 rows).
This creates a new immutable **Snapshot 2**, advancing the current table pointer while
preserving **Snapshot 1** in its exact historical state.

In [ ]:
# Take a slice of 1,000 rows as a second batch
second_batch = arrow_table.slice(0, 1000)
print(f"Appending batch 2: {second_batch.num_rows} rows...")

time.sleep(2)  # Ensure distinct commit timestamp
table.append(second_batch)

current_snap = table.current_snapshot()
snapshot_2_id = current_snap.snapshot_id
snapshot_2_timestamp = current_snap.timestamp_ms
snapshot_2_dt = datetime.utcfromtimestamp(snapshot_2_timestamp / 1000).isoformat()

print(f"Snapshot 2 committed successfully:")
print(f"  Snapshot 2 ID : {snapshot_2_id}")
print(f"  Timestamp     : {snapshot_2_timestamp} ({snapshot_2_dt} UTC)")
print(f"  Parent ID     : {current_snap.parent_snapshot_id} (matches Snapshot 1: {snapshot_1_id})")

### 10. Execute Time-Travel Queries in Trino
We now query the table across three distinct temporal states:
1. **Head (Current Table State):** Returns 2,964,624 + 1,000 = 2,965,624 rows.
2. **Time-Travel by Snapshot ID (`FOR VERSION AS OF`):** Queries Snapshot 1; returns exactly 2,964,624 rows.
3. **Time-Travel by Timestamp (`FOR TIMESTAMP AS OF`):** Queries table state at Snapshot 1 commit time; returns exactly 2,964,624 rows.

In [ ]:
# 1. Query current head state
cur.execute(f'SELECT COUNT(*) FROM "{NAMESPACE}"."{TABLE_NAME}"')
head_count = cur.fetchone()[0]
print(f"[Current Head State] Total rows: {head_count:,} (Expected: 2,965,624)")
assert head_count == 2965624

# 2. Time-Travel Query by Version (Snapshot ID)
time_travel_version_query = f'''
SELECT COUNT(*) 
FROM "{NAMESPACE}"."{TABLE_NAME}" FOR VERSION AS OF {snapshot_1_id}
'''
cur.execute(time_travel_version_query)
snap1_count = cur.fetchone()[0]
print(f"[Time-Travel: FOR VERSION AS OF {snapshot_1_id}] Rows: {snap1_count:,} (Expected: 2,964,624)")
assert snap1_count == 2964624

# 3. Time-Travel Query by Timestamp (Between Snapshot 1 and Snapshot 2)
# We query at timestamp 1 second after Snapshot 1 commit
query_ts = datetime.utcfromtimestamp((snapshot_1_timestamp + 1000) / 1000).strftime('%Y-%m-%d %H:%M:%S.000 UTC')
time_travel_ts_query = f'''
SELECT COUNT(*) 
FROM "{NAMESPACE}"."{TABLE_NAME}" FOR TIMESTAMP AS OF TIMESTAMP '{query_ts}'
'''
cur.execute(time_travel_ts_query)
snap_ts_count = cur.fetchone()[0]
print(f"[Time-Travel: FOR TIMESTAMP AS OF '{query_ts}'] Rows: {snap_ts_count:,} (Expected: 2,964,624)")
assert snap_ts_count == 2964624

print("\n>>> Time-travel assertions PASSED: historical isolation verified across versions and timestamps!")

### 11. In-Place Non-Destructive Schema Evolution
A key advantage of Apache Iceberg is **in-place schema evolution**:
- Columns can be added, renamed, or reordered **without rewriting existing Parquet data files** ($O(1)$ metadata operation).
- Existing Parquet files are read with `NULL` for newly added columns.
- Column identity is tracked by unique numeric IDs, not by column names, eliminating column name collision bugs.

In [ ]:
# 1. Add new column 'surge_multiplier' to Iceberg table schema
print("Evolving schema: Adding 'surge_multiplier' (Double)...")
with table.update_schema() as update:
    update.add_column("surge_multiplier", DoubleType(), doc="Dynamic surge pricing factor")

# Verify new column appears in table schema
evolved_schema = table.schema()
print(f"New column present in Iceberg schema: {'surge_multiplier' in [f.name for f in evolved_schema.fields]}")

# 2. Query evolved schema via Trino: historical records automatically return NULL
cur.execute(f'''
SELECT VendorID, fare_amount, surge_multiplier 
FROM "{NAMESPACE}"."{TABLE_NAME}" 
LIMIT 5
''')
evolved_sample = pd.DataFrame(cur.fetchall(), columns=["VendorID", "fare_amount", "surge_multiplier"])
print("\n=== Historical Records Read with Evolved Schema (NULL surge_multiplier) ===")
print(evolved_sample)
assert evolved_sample["surge_multiplier"].isna().all()

# 3. Rename an existing column: rename 'Airport_fee' -> 'airport_surcharge'
print("\nEvolving schema: Renaming 'Airport_fee' -> 'airport_surcharge'...")
with table.update_schema() as update:
    update.rename_column("Airport_fee", "airport_surcharge")

# Query renamed column via Trino
cur.execute(f'''
SELECT VendorID, airport_surcharge 
FROM "{NAMESPACE}"."{TABLE_NAME}" 
WHERE airport_surcharge > 0
LIMIT 5
''')
renamed_sample = pd.DataFrame(cur.fetchall(), columns=["VendorID", "airport_surcharge"])
print("\n=== Renamed Column 'airport_surcharge' Query Result ===")
print(renamed_sample)

print("\n>>> Schema evolution verified: columns added and renamed with ZERO data rewrites!")

### 12. Lakehouse Table Maintenance (Snapshot Expiration)
As new snapshots are created, old historical snapshots accumulate metadata and unreferenced files.
Here we demonstrate programmatic snapshot expiration, retaining only the latest snapshot.

In [ ]:
print(f"Snapshots before maintenance: {len(table.snapshots())}")

# Expire snapshots older than current timestamp, retaining only the latest snapshot
cutoff_time = int(time.time() * 1000)
try:
    # Trigger snapshot expiration via Trino table procedure
    cur.execute(f'ALTER TABLE "{NAMESPACE}"."{TABLE_NAME}" EXECUTE expire_snapshots(retention_threshold => \'1s\')')
    result = cur.fetchall()
    print(f"Trino snapshot expiration result: {result}")
except Exception as e:
    print(f"Trino expire procedure notice ({e}); verifying PyIceberg snapshot state...")

# Refresh table metadata
table.refresh()
remaining_snapshots = table.snapshots()
print(f"Snapshots remaining after expiration: {len(remaining_snapshots)}")
print(f"Current active snapshot: {table.current_snapshot().snapshot_id}")

### 13. Superset BI Metadata Synchronization
Register the newly partitioned, evolved table into Superset for analytics dashboards.

In [ ]:
from superset import SupersetConfigurator
superset_url = "http://superset:8088"

dashboard_name = f"LAKEHOUSE_POC.{NAMESPACE}.{TABLE_NAME}"
chart_name = f"{NAMESPACE}.{TABLE_NAME}"

configurator = SupersetConfigurator(superset_url)
print("Configuring Superset dataset and dashboard...")

dataset = configurator.get_dataset(schema_name=NAMESPACE, table_name=TABLE_NAME)
if dataset is None:
    dataset = configurator.create_dataset(schema_name=NAMESPACE, table_name=TABLE_NAME)
    print("Superset dataset created successfully!")

dashboard = configurator.get_dashboard(dashboard_name)
if dashboard is None:
    dashboard = configurator.create_dashboard(dashboard_name)
    print(f"Superset dashboard created: {dashboard_name}")

chart = configurator.get_table_chart(chart_name)
if chart is None:
    chart = configurator.create_table_chart(name=chart_name, dataset_id=dataset["id"])
    print(f"Superset chart created: {chart_name}")
    configurator.add_table_chart_to_dashboard(dashboard, chart)
    print("Chart attached to dashboard successfully.")

print("\n Lakehouse demonstration completed with 100% success!")